# EDA: Maternal Health Risk Data Set

**Автор:** Селезнева Алиса, магистратура 2 курс  
**Курс:** MLOps  
**Датасет:** Maternal Health Risk (UCI ML Repository), 1014 наблюдений, 6 признаков, 3 класса риска.

## Описание признаков
- **Age** — возраст беременной в годах.
- **SystolicBP** — систолическое давление (верхнее), мм рт. ст.
- **DiastolicBP** — диастолическое давление (нижнее), мм рт. ст.
- **BS** — уровень глюкозы в крови, ммоль/л.
- **BodyTemp** — температура тела, °F.
- **HeartRate** — пульс, уд/мин.
- **RiskLevel** — уровень риска: `low risk`, `mid risk`, `high risk`.

## Цель исследования
1. Понять структуру данных и распределения.
2. Оценить баланс классов.
3. Выявить связи между признаками.
4. Обнаружить выбросы (клинически значимые).
5. Подготовить почву для feature engineering.

In [2]:
import sys
import os
from pathlib import Path

# Добавляем корень проекта в sys.path,
# чтобы импорты из src/ работали внутри ноутбука
PROJECT_ROOT = Path(os.getcwd()).parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import warnings
import numpy as np
import pandas as pd

from src.data.loader import load_data
from src.features.preprocessing import Winsorizer
from src.visualization.plots import (
    plot_numeric_distributions,
    plot_class_balance,
    plot_phik_correlation,
    plot_boxplots,
)

warnings.filterwarnings("ignore")
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)
print("Импорты готовы")
print(f"PROJECT_ROOT = {PROJECT_ROOT}")

Импорты готовы
PROJECT_ROOT = c:\Users\Alice\projects\mlops_maternal_health


## 1. Загрузка данных

In [3]:
df = load_data()

print(f"Размер датасета: {df.shape}")
print(f"Столбцы: {list(df.columns)}")
df.head()

Размер датасета: (1014, 7)
Столбцы: ['Age', 'SystolicBP', 'DiastolicBP', 'BS', 'BodyTemp', 'HeartRate', 'RiskLevel']


,Age,SystolicBP,DiastolicBP,BS,BodyTemp,HeartRate,RiskLevel
0,25,130,80,15.0,98.0,86,high risk
1,35,140,90,13.0,98.0,70,high risk
2,29,90,70,8.0,100.0,80,high risk
3,30,140,85,7.0,98.0,70,high risk
4,35,120,60,6.1,98.0,76,low risk


In [4]:
print("--- Типы данных и пропуски ---")
df.info()

print("\n--- Пропуски ---")
print(df.isnull().sum())

print("\n--- Описательная статистика ---")
df.describe().round(2)

--- Типы данных и пропуски ---
<class 'pandas.DataFrame'>
RangeIndex: 1014 entries, 0 to 1013
Data columns (total 7 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   Age          1014 non-null   int64  
 1   SystolicBP   1014 non-null   int64  
 2   DiastolicBP  1014 non-null   int64  
 3   BS           1014 non-null   float64
 4   BodyTemp     1014 non-null   float64
 5   HeartRate    1014 non-null   int64  
 6   RiskLevel    1014 non-null   str    
dtypes: float64(2), int64(4), str(1)
memory usage: 55.6 KB

--- Пропуски ---
Age            0
SystolicBP     0
DiastolicBP    0
BS             0
BodyTemp       0
HeartRate      0
RiskLevel      0
dtype: int64

--- Описательная статистика ---


,Age,SystolicBP,DiastolicBP,BS,BodyTemp,HeartRate
count,1014.00,1014.0,1014.00,1014.00,1014.00,1014.00
mean,29.87,113.2,76.46,8.73,98.67,74.30
std,13.47,18.4,13.89,3.29,1.37,8.09
min,10.00,70.0,49.00,6.00,98.00,7.00
25%,19.00,100.0,65.00,6.90,98.00,70.00
50%,26.00,120.0,80.00,7.50,98.00,76.00
75%,39.00,120.0,90.00,8.00,98.00,80.00
max,70.00,160.0,100.00,19.00,103.00,90.00


### Вывод по разделу
- В датасете **1014 строк и 7 столбцов**: 6 клинических признаков + целевая переменная.
- **Пропусков нет** — данные чистые.
- Признаки имеют **разный масштаб** (Age: 10–70, BS: 6–19) — потребуется масштабирование для линейных моделей.
- Все признаки числовые, целевая переменная — строковая (3 класса).

## 2. Распределения числовых признаков

In [5]:
numeric_cols = df.select_dtypes(include=np.number).columns.tolist()
print(f"Числовые признаки: {numeric_cols}")

fig = plot_numeric_distributions(df, numeric_cols)
fig.show()

Числовые признаки: ['Age', 'SystolicBP', 'DiastolicBP', 'BS', 'BodyTemp', 'HeartRate']


### Интерпретация
- **Age**: скошено влево, преобладают молодые женщины, есть значения до 70 лет.
- **SystolicBP / DiastolicBP**: бимодальные распределения — вероятно, подгруппы пациенток.
- **BS**: большинство в диапазоне 6–8 ммоль/л, но есть выбросы до 19 — клинически значимы.
- **BodyTemp и HeartRate**: близки к нормальному распределению.

## 3. Баланс классов

In [6]:
counts = df["RiskLevel"].value_counts()
print("Распределение классов:")
print(counts)
print(f"\nДоли:")
print((counts / counts.sum()).round(3))
print(f"\nСоотношение min/max: {counts.min() / counts.max():.2f}")

fig = plot_class_balance(df, "RiskLevel")
fig.show()

Распределение классов:
RiskLevel
low risk     406
mid risk     336
high risk    272
Name: count, dtype: int64

Доли:
RiskLevel
low risk     0.400
mid risk     0.331
high risk    0.268
Name: count, dtype: float64

Соотношение min/max: 0.67


### Вывод по балансу классов
- Классы **несбалансированы**: low risk ≈ 40%, high risk ≈ 32%, mid risk ≈ 28%.
- Соотношение min/max ≈ 0.70 — **умеренный дисбаланс**.
- Это обосновывает выбор **macro F1-score** как основной метрики: метрика одинаково важна для всех классов и не позволяет модели игнорировать меньшинство.

## 4. Корреляционный анализ (Phik)

In [7]:
interval_cols = df.select_dtypes(include=np.number).columns.tolist()
fig = plot_phik_correlation(df, interval_cols)
fig.show()

### Вывод по корреляциям
Наибольшая связь с `RiskLevel`:
- **BS** (~0.50) — уровень глюкозы.
- **SystolicBP** (~0.45) — систолическое давление.
- **DiastolicBP** (~0.42) — диастолическое давление.
- **Age** (~0.25).

Это согласуется с медицинскими представлениями: гипергликемия и гипертензия — ключевые факторы риска при беременности.

**Почему Phik, а не Pearson?** Phik улавливает **нелинейные** зависимости и работает со **смешанными** типами данных.

## 5. Анализ выбросов

In [8]:
fig = plot_boxplots(df, numeric_cols)
fig.show()

In [9]:
winsorizer = Winsorizer(cols=numeric_cols, lower_q=0.01, upper_q=0.99)
winsorizer.fit(df)
bounds = winsorizer.get_bounds()

outlier_info = pd.DataFrame({
    col: {
        "lower_1pct": bounds[col][0],
        "upper_99pct": bounds[col][1],
        "min_raw": df[col].min(),
        "max_raw": df[col].max(),
    }
    for col in numeric_cols
}).T

outlier_info.round(2)

,lower_1pct,upper_99pct,min_raw,max_raw
Age,12.0,60.0,10.0,70.0
SystolicBP,75.0,140.0,70.0,160.0
DiastolicBP,49.0,100.0,49.0,100.0
BS,6.0,19.0,6.0,19.0
BodyTemp,98.0,103.0,98.0,103.0
HeartRate,60.0,90.0,7.0,90.0


### Вывод по выбросам
- **BS, SystolicBP, DiastolicBP, HeartRate** содержат выбросы.
- **Полное удаление выбросов недопустимо** в медицинской задаче: высокие значения BS или давления — это и есть **сигнал риска**.
- Применяем **winsorization** (ограничение по 1-му и 99-му перцентилю): сохраняет информацию о «крайности» значения, но ограничивает влияние на модель.
- **Ключевой момент:** winsorization должна фититься **только на train**, иначе возникает **data leakage**. Это реализовано в `src/features/preprocessing.py::Winsorizer`.

## 6. Итоги EDA

### Что мы узнали о данных
1. **1014 наблюдений, 6 признаков, 3 класса** — компактный, но информативный датасет.
2. **Пропусков нет** — чистая работа с данными.
3. **Дисбаланс классов** — используем macro F1.
4. **Ключевые признаки** — BS, SystolicBP, DiastolicBP.
5. **Выбросы клинически значимы** — не удаляем, а ограничиваем (winsorization).

### Что делаем дальше
- **Feature engineering** — PulsePressure, MAP, категории возраста и глюкозы (`src/features/build.py`).
- **Preprocessing** — winsorization только на train (`src/features/preprocessing.py::Winsorizer`).
- **Обучение 3 моделей** — LR, RF, CatBoost (`src/models/train.py`).
- **Оценка** — матрицы ошибок, метрики F1 на train/test.

### Ключевой принцип проекта
> Вся логика обработки данных и обучения вынесена в модули `src/`. Этот ноутбук — только исследование, он вызывает готовые функции. Это позволяет переиспользовать код в продакшне.